## Get backscatter timeseries, generate plots for each station-year

Elements needed for this script:
* SNOTEL station coordinates (CSV)
* Peak SWE for each SNOTEL station-year (CSV)*
* 50% Peak SWE thresholds (CSV)*
* SNOTEL station data, processed (CSV)*

Note*: Items marked with an asterisk can be obtained using scripts found here: https://github.com/allydetre/snotelprocessr/tree/main/examples

In [ ]:
# import packages
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import geopandas as gpd
import rasterio as rio
import glob
import re
import xarray as xr
import rioxarray as rxr
from rasterio.plot import show
from pyproj import Proj
from pyproj import Transformer
from shapely.geometry import box
from scipy import stats
from rasterio.warp import calculate_default_transform, reproject, Resampling
import gc

sys.path.append('../sar_snowmelt_timing')
import s1_rtc_bs_utils

### Set up folder paths for DEM, GEOJSONs

In [ ]:
geojson_dir = "../import/geojsons/" # Geojson folder - lat/lon
dem_dir = "../import/USGS3DEP_DEM/" # Import folder with generated DEMs

### Initialize data needed for Sentinel-1 backscatter timeseries/plot generation:

In [ ]:
# SNOTEL coordinates
snotelcoords = pd.read_csv('../import/df/CO_testsubset.csv')

# Import peak SWE
peak_swe_df = pd.read_csv("../import/df/CO_testsubset_peakswe.csv")

# Import snow free dates df
swe50p = pd.read_csv("../import/df/CO_testsubset_peakswewindow.csv")

# Define SNOTEL data folder with SWE, temps
swe_folder = "../import/df/processed_snotel"

# output csv path
csv_output_path = "../output/df/CO_testsubset_backscatter.csv"

# Figure output path
plot_output_dir = "../output/backscatter_plots/CO_testsubset"



epsg_code = None  # Initialize EPSG code

csv_data = []  # List to store CSV output data

### What the following for loop does:
1. Grabs Sentinel-1 SAR imagery over defined time period/bounding box
2. Masks imagery by worldcover classes (excludes dense vegetation, humanmade objects, bodies of water, bare ground) and elevation (+/- 100 m from SNOTEL station elevation)
3. Grabs all Sentinel-1 SAR backscatter observations within 50% peak SWE window for all orbits/passes - appends to a CSV
4. Plots Sentinel-1 SAR backscatter timeseries with SNOTEL measurements of air temperature (daily max), SWE

In [ ]:
# Process each GeoJSON file
geojson_files = glob.glob(os.path.join(geojson_dir, "*.geojson"))
for geojson_file in geojson_files:
    print(f'Processing {geojson_file}')
    bbox_gdf = gpd.read_file(geojson_file)
    station_name = os.path.basename(geojson_file).split('.')[0]
    state, site = station_name.split('_')
    
    coords = snotelcoords[(snotelcoords['state'].str.strip() == state) & 
                          (snotelcoords['site'].astype(str).str.strip() == site)]
    
    if not coords.empty:
        x, y = coords[['easting_exact', 'northing_exact']].values[0]
        epsg_code = int(coords['epsg'].values[0])
    else:
        print(f'No coordinates found for {station_name}. Skipping...')
        continue
    
    start_time = '2014-01-01'
    end_time = '2024-08-01'
    
    ts_ds = s1_rtc_bs_utils.get_s1_rtc_stac_pc(bbox_gdf, start_time, end_time, 'all', 20, epsg_code)
    worldcover = s1_rtc_bs_utils.get_worldcover(ts_ds)
    ts_ds_clip = ts_ds.where(worldcover.isin([20, 30, 40, 70, 90, 95, 100])) # excludes dense vegetation, humanmade objects, bodies of water, bare ground

    # Load in DEM
    dem_file = os.path.join(dem_dir, f"{station_name}_3DEP_DEM.tif") 
    if not os.path.exists(dem_file):
        print(f'DEM file not found for {station_name}. Skipping elevation masking...')
        continue
    
    DEM = rxr.open_rasterio(dem_file).rio.reproject_match(ts_ds_clip)
    elevation = DEM.sel(x=x, y=y, method="nearest").values
    elevation_mask = (DEM[0] >= elevation - 100) & (DEM[0] <= elevation + 100)
    ts_ds_clip = ts_ds_clip.where(elevation_mask, np.nan)
    
    years = [2015, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024] # 2016 excluded due to gaps

    if ts_ds_clip.time.size == 0:
        print(f"No Sentinel-1 data available for station {station_name}. Skipping.")
        continue  # Skip to the next station
    
    for year in years:
        peak_swe_df['station_name'] = peak_swe_df['station_name'].str.strip()
        peak_swe_df['year'] = peak_swe_df['year'].astype(int)
        filtered_swe_df = peak_swe_df[(peak_swe_df['station_name'] == station_name) & (peak_swe_df['year'] == year)]
        
        f, ax = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
        f.suptitle(f'Backscatter Time Series for {station_name}, {year}')

        # Second axis for SWE
        ax2_vv = ax[0].twinx()
        ax2_vh = ax[1].twinx()

        # Create third y-axis for temperature (daily max)
        ax3_vv = ax2_vv.twinx()  
        ax3_vh = ax2_vh.twinx()

        # Offset temperature axis to the right so it doesn’t overlap with SWE axis
        ax3_vv.spines["right"].set_position(("outward", 60))  
        ax3_vh.spines["right"].set_position(("outward", 60))

        # Load SWE/temperature data for this station
        swe_file = os.path.join(swe_folder, f"{station_name}_processed.csv")
        
        if os.path.exists(swe_file):
            swe_df = pd.read_csv(swe_file, parse_dates=['date'])
            swe_df = swe_df[swe_df['date'].dt.year == year]
            
            if not swe_df.empty:
                # Plot SWE time series
                swe_dates = swe_df['date']
                swe_values = swe_df['swe']
                ax2_vv.plot(swe_dates, swe_values, label='SWE', color='#0c9a95', linestyle='-', linewidth=2, zorder = 1)
                ax2_vh.plot(swe_dates, swe_values, label='SWE', color='#0c9a95', linestyle='-', linewidth=2, zorder = 1)
                
                # Plot temperature time series using maximum daily airtemp from SNOTEL
                ax3_vv.plot(swe_df['date'], swe_df['airtemp_max'], label='Air Temp', color='#6e90af', linestyle='-', linewidth=2, alpha=0.7)
                ax3_vh.plot(swe_df['date'], swe_df['airtemp_max'], label='Air Temp', color='#6e90af', linestyle='-', linewidth=2, alpha=0.7)

        
        swe50p_row = swe50p[(swe50p['station_name'] == station_name) & (swe50p['year'] == year)]
        
        if not swe50p_row.empty:
            first_50p_swe = int(swe50p_row['first_50p_swe'].values[0])
            last_50p_swe = int(swe50p_row['last_50p_swe'].values[0])
            
            first_50p_swe_date = pd.to_datetime(f"{year}-{first_50p_swe}", format="%Y-%j")
            last_50p_swe_date = pd.to_datetime(f"{year}-{last_50p_swe}", format="%Y-%j")
            
            ts_ds_clip_restricted = ts_ds_clip.loc[slice(first_50p_swe_date, last_50p_swe_date)].dropna('time', how='all')
        else:
            ts_ds_clip_restricted = ts_ds_clip.loc[slice(f"{year}-01-01", f"{year}-08-01")].dropna('time', how='all')

        # Skip this year if no data available       
        if ts_ds_clip_restricted.time.values.size == 0:
            print(f"Skipping year {year}: No valid data found.")
            continue 
        
        orbits = s1_rtc_bs_utils.get_orbits_with_melt_season_coverage(ts_ds_clip_restricted, 1)
        ts_ds_clip_restricted = ts_ds_clip_restricted[ts_ds_clip_restricted['sat:relative_orbit'].isin(orbits)]
        
        colors1 = iter(['#2e1561', '#1c2c7c', '#366ea9', '#6cafca', '#b8dfe8'])
        colors2 = iter(['#ac0e0e', '#c25106', '#cc701d', '#dba647', '#f2d064'])
        
        for orbit in np.unique(ts_ds_clip_restricted['sat:relative_orbit']):
            single_orbit_full = ts_ds_clip[ts_ds_clip['sat:relative_orbit'] == orbit].median(dim=['x', 'y'])
            single_orbit_restricted = ts_ds_clip_restricted[ts_ds_clip_restricted['sat:relative_orbit'] == orbit].median(dim=['x', 'y'])
            
            if single_orbit_restricted.isnull().all():
                continue
            
            single_orbit_db = 10 * np.log10(single_orbit_full)
            asc_or_desc = single_orbit_full['sat:orbit_state'][0].values
            
            # Labeling for legend - ascending vs descending
            orbit_direction = "(A)" if asc_or_desc == "ascending" else "(D)"
            color = next(colors1) if asc_or_desc == 'ascending' else next(colors2)
            
            time = single_orbit_db['time'].values
            vv_data = single_orbit_db.sel(band='vv').values
            vh_data = single_orbit_db.sel(band='vh').values
            
            vv_valid = single_orbit_restricted.sel(band='vv').dropna('time')
            vh_valid = single_orbit_restricted.sel(band='vh').dropna('time')

            # Use only the restricted (filtered) dataset for export
            single_orbit_restricted_db = 10 * np.log10(single_orbit_restricted)
            time_filtered = single_orbit_restricted_db['time'].values
            vv_data_filtered = single_orbit_restricted_db.sel(band='vv').values
            vh_data_filtered = single_orbit_restricted_db.sel(band='vh').values
            
            # Store only the filtered backscatter values used in the plot
            for t, vv, vh in zip(time_filtered, vv_data_filtered, vh_data_filtered):
                doy = pd.to_datetime(str(t)).dayofyear
                csv_data.append([station_name, year, 'vv', orbit, asc_or_desc, doy, vv])
                csv_data.append([station_name, year, 'vh', orbit, asc_or_desc, doy, vh])
            
            vv_estimate = vv_valid.idxmin().values if not vv_valid.isnull().all() else None
            vh_estimate = vh_valid.idxmin().values if not vh_valid.isnull().all() else None
            
            if vv_estimate:
                ax[0].axvline(x=vv_estimate, linestyle='dashed', color=color, zorder = 4)
            
            if vh_estimate:
                ax[1].axvline(x=vh_estimate, linestyle='dashed', color=color, zorder = 4)
            
            ax[0].plot(time, vv_data, label=f'Orbit {orbit} VV {orbit_direction}', linestyle='-', marker='o', color=color, zorder = 3)
            ax[1].plot(time, vh_data, label=f'Orbit {orbit} VH {orbit_direction}', linestyle='-', marker='o', color=color, zorder = 3)
        
        if not filtered_swe_df.empty:
            doy = filtered_swe_df['peak_swe'].values[0]
            peak_swe_date = pd.to_datetime(f'{year}-{doy}', format='%Y-%j')
            ax[0].axvline(x=peak_swe_date, linestyle='solid', color='#12b7b1', linewidth=2, label='Peak SWE', zorder = 2)
            ax[1].axvline(x=peak_swe_date, linestyle='solid', color='#12b7b1', linewidth=2, label='Peak SWE', zorder = 2)
        
        ax2_vv.set_zorder(0)
        ax[0].set_zorder(2)  # Ensure backscatter is in front
        ax[0].patch.set_visible(False)  # Make sure background isn't hiding anything
        ax2_vh.set_zorder(0)
        ax[1].set_zorder(2)
        ax[1].patch.set_visible(False)
        ax3_vv.set_zorder(0)
        ax3_vh.set_zorder(0)

        # x lims
        ax[0].set_xlim(pd.to_datetime(f'{year}-01-01'), pd.to_datetime(f'{year}-07-02')) # jan to july window
        ax[1].set_xlim(pd.to_datetime(f'{year}-01-01'), pd.to_datetime(f'{year}-07-02')) # jan to july window

        # peak SWE
        ax[0].axvspan(first_50p_swe_date, last_50p_swe_date, color='#d2d2d2', alpha=0.3, label='50% Peak SWE', zorder = 0)
        ax[1].axvspan(first_50p_swe_date, last_50p_swe_date, color='#d2d2d2', alpha=0.3, label='50% Peak SWE', zorder = 0)

        # labeling secondary axes
        ax[0].set_ylabel("Backscatter (dB)", color='black')
        ax[1].set_ylabel("Backscatter (dB)", color='black')
        ax2_vv.set_ylabel("SWE (cm)", color='#0c9a95')
        ax2_vh.set_ylabel("SWE (cm)", color='#0c9a95')
        
        # Move the SWE axis to the right side
        ax2_vv.yaxis.set_label_position("right")
        ax2_vv.yaxis.tick_right()
        ax2_vv.spines["right"].set_position(("outward", 0))
        
        # Move temp axis to right right side
        ax2_vh.yaxis.set_label_position("right")
        ax2_vh.yaxis.tick_right()
        ax2_vh.spines["right"].set_position(("outward", 0))
        ax3_vv.set_ylabel("Air Temperature (Daily Max, °C)", color='#6e90af')
        ax3_vh.set_ylabel("Air Temperature (Daily Max, °C)", color='#6e90af')

        # Add horizontal dashed line for 0 deg C temperature
        ax3_vv.axhline(y=0, color='gray', linestyle='dashed', label = "0°C", linewidth=1, alpha=0.5, zorder=0)
        ax3_vh.axhline(y=0, color='gray', linestyle='dashed', label = "0°C", linewidth=1, alpha=0.5, zorder=0)

        # legends
        # ax[0].legend(loc='center', bbox_to_anchor=(0.5, -0.15), ncol=4, fontsize=10)
        # ax[1].legend(loc='center', bbox_to_anchor=(0.5, -0.15), ncol=4, fontsize=10)
        temp_line = ax3_vv.axhline(y=0, color='gray', linestyle='dashed', label="0°C", linewidth=1, alpha=0.5, zorder=0)
        ax[0].legend(handles=[*ax[0].get_legend_handles_labels()[0], temp_line], 
                     loc='center', bbox_to_anchor=(0.5, -0.16), ncol=5, fontsize=10)
        ax[1].legend(handles=[*ax[1].get_legend_handles_labels()[0], temp_line], 
                     loc='center', bbox_to_anchor=(0.5, -0.16), ncol=5, fontsize=10)

        plot_filename = f"{station_name}_dBtimeseries_{year}.png"
        plt.savefig(os.path.join(plot_output_dir, plot_filename), dpi=300, bbox_inches='tight')
        plt.close()

    # Free memory after processing current GeoJSON
    for var in ["ts_ds_clip", "ts_ds_clip_restricted", "DEM", "single_orbit_db"]:
        if var in globals():
            del globals()[var]

# Save CSV data
csv_df = pd.DataFrame(csv_data, columns=['station_name', 'year', 'band', 'orbit', 'orbit_direction', 'backscatter_DOY', 'backscatter_value'])
csv_df = csv_df.dropna(subset=['backscatter_value']) # drop blanks
csv_df.to_csv(csv_output_path, index=False)
print(f"CSV saved: {csv_output_path}")